# 问题三 Baseline：Direct Multi-step Hybrid Model

本 notebook 是第三问的基础框架 baseline。

第三问任务：

```text
结合质量守恒 / 水力停留时间思想与数据驱动方法，
预测未来 1–12 小时的出厂水浊度 NTU，
并给出 2026-02-01、2026-02-10、2026-02-20 的 07:00–19:00 预测结果。
```

由于原始数据每 2 小时记录一次，本 notebook 将未来 1–12 小时预测离散化为：

```text
未来 2h, 4h, 6h, 8h, 10h, 12h
```

即 6 个 horizon。

---

## 本 baseline 的核心结构

```text
merged.xlsx
↓
DATETIME / OP_DATE
↓
物理代理特征
HRT_PROXY, FLOW_RATIO, WELL_LEVEL_CHANGE
↓
历史 lag 特征
NTU_lag1-lag12 + process variables lag0-lag12
↓
未来多步目标
NTU_t+2h ... NTU_t+12h
↓
Direct Multi-output Models
Ridge / Random Forest / XGBoost
↓
模型验证
MAE / RMSE / R² by horizon
↓
目标日期 07:00–19:00 预测
↓
敏感性分析
R/W NTU +20%, ALUM +10%
```

---

## 输出目录

```text
outputs/problem3_baseline/
```

重点输出文件：

```text
problem3_baseline_model_results.xlsx
problem3_baseline_horizon_metrics.xlsx
problem3_baseline_test_predictions.xlsx
problem3_baseline_target_7_19_predictions.xlsx
problem3_baseline_sensitivity_analysis.xlsx
figures/
models/
```

本 notebook 不包含 GRU。  
它的作用是先建立一个稳定、可解释、可提交的第三问基础版本。


## 1. 导入依赖库

本 baseline 使用传统机器学习模型做多步预测。  
为了兼容旧版 scikit-learn，RMSE 使用：

```python
np.sqrt(mean_squared_error(y_true, y_pred))
```


In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as e:
    XGB_AVAILABLE = False
    XGB_IMPORT_ERROR = e

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

print("XGBoost available:", XGB_AVAILABLE)
if not XGB_AVAILABLE:
    print("XGBoost import error:", XGB_IMPORT_ERROR)


## 2. 参数设置

目标变量：

```text
NTU
```

预测 horizon：

```text
2h, 4h, 6h, 8h, 10h, 12h
```

统一时间顺序划分：

```text
前 70% train
后 30% test
```


In [ ]:
TARGET_COL = "NTU"
OUTPUT_DIR_NAME = "problem3_baseline"

RANDOM_STATE = 42

TRAIN_RATIO = 0.70
RECORD_INTERVAL_HOURS = 2

# 未来 2–12 小时，共 6 个 horizon
FORECAST_STEPS = [1, 2, 3, 4, 5, 6]
HORIZON_HOURS = [step * RECORD_INTERVAL_HOURS for step in FORECAST_STEPS]

# 历史 lag
NTU_LAGS = list(range(1, 13))       # NTU_lag1-lag12，不使用 NTU_lag0 防止泄漏
PROCESS_LAGS = list(range(0, 13))   # 过程变量可使用 lag0-lag12

# 目标日期与时间
TARGET_DATES = [
    "2026-02-01",
    "2026-02-10",
    "2026-02-20",
]

TARGET_HOURS = [7, 9, 11, 13, 15, 17, 19]

# 是否 clipping 目标 NTU
# 第三题 baseline 默认不 clipping，保持真实预测目标。
CLIP_NTU = False
NTU_CLIP_UPPER = 2.0

# 敏感性分析设置
RAW_NTU_SHOCK_RATE = 0.20     # R/W NTU +20%
ALUM_ADJUST_RATE = 0.10       # ALUM +10%

print("参数设置完成。")


## 3. 自动定位 `merged.xlsx`

本 notebook 从原始 `merged.xlsx` 读取。  
不要使用标准化、clipping 或 selected-lag 中间文件。


In [ ]:
def locate_merged_file(filename="merged.xlsx"):
    cwd = Path.cwd().resolve()
    candidates = []

    search_roots = [cwd] + list(cwd.parents)

    for root in search_roots:
        candidates.extend([
            root / "data" / filename,
            root / "codes" / "data" / filename,
            root / "2026-Asia-Pacific-cup" / "data" / filename,
            root / "2026-Asia-Pasific-cup" / "data" / filename,
            root / filename,
        ])

    seen = set()
    unique_candidates = []
    for p in candidates:
        p = p.resolve()
        if p not in seen:
            seen.add(p)
            unique_candidates.append(p)

    for p in unique_candidates:
        if p.exists():
            return p

    for p in cwd.rglob(filename):
        return p.resolve()

    for parent in cwd.parents:
        try:
            for p in parent.rglob(filename):
                return p.resolve()
        except Exception:
            pass

    searched = "\n".join(str(p) for p in unique_candidates)
    raise FileNotFoundError(f"未找到 {filename}。已检查路径：\n{searched}")


DATA_PATH = locate_merged_file()
DATA_DIR = DATA_PATH.parent

if DATA_DIR.name == "data":
    PROJECT_DIR = DATA_DIR.parent
else:
    PROJECT_DIR = DATA_DIR

OUTPUT_DIR = PROJECT_DIR / "outputs" / OUTPUT_DIR_NAME
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("当前工作目录：", Path.cwd().resolve())
print("使用数据文件：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)


## 4. 读取数据并构造时间列

构造：

```text
DATETIME
OP_DATE
```

其中 `OP_DATE` 使用运行日定义：

```text
07:00 至次日 05:00 归为同一个运行日。
```


In [ ]:
df = pd.read_excel(DATA_PATH)

print("原始数据规模：", df.shape)
print("原始列名：")
print(df.columns.tolist())

if TARGET_COL not in df.columns:
    raise ValueError(f"未找到目标变量 {TARGET_COL}。请检查 merged.xlsx。")


def construct_datetime(data):
    data = data.copy()

    if "DATETIME" in data.columns:
        data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
        return data

    date_candidates = ["DATE", "Date", "date"]
    time_candidates = ["TIME", "Time", "time"]

    date_col = next((c for c in date_candidates if c in data.columns), None)
    time_col = next((c for c in time_candidates if c in data.columns), None)

    if date_col is None or time_col is None:
        raise ValueError("无法构造 DATETIME：需要 DATE 和 TIME 两列，或已有 DATETIME 列。")

    date_text = data[date_col].astype(str).str.split().str[0]
    time_text = data[time_col].astype(str).str.split().str[-1]

    data["DATETIME"] = pd.to_datetime(
        date_text + " " + time_text,
        errors="coerce",
    )

    return data


df = construct_datetime(df)
df = df.dropna(subset=["DATETIME"]).sort_values("DATETIME").reset_index(drop=True)

df["OP_DATE"] = df["DATETIME"].dt.date
mask_before_7 = df["DATETIME"].dt.hour < 7

df.loc[mask_before_7, "OP_DATE"] = (
    df.loc[mask_before_7, "DATETIME"] - pd.Timedelta(days=1)
).dt.date

print("时间范围：", df["DATETIME"].min(), "至", df["DATETIME"].max())
display(df[["DATETIME", "OP_DATE", TARGET_COL]].head(15))
display(df[["DATETIME", "OP_DATE", TARGET_COL]].tail(15))


## 5. 基础清洗与泵状态转换

处理内容：

```text
1. 数值列转 numeric；
2. F/RIDE 缺失填 0；
3. R/W PUMP DUTY、T/W PUMP DUTY 转换为泵数量；
4. NTU 默认不 clipping。
```


In [ ]:
def pump_duty_to_count(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s == "":
        return np.nan

    if s.endswith(".0"):
        s = s.replace(".0", "")

    if "+" in s:
        parts = [p.strip() for p in s.split("+") if p.strip() != ""]
        return len(parts)

    if s.isdigit():
        return 1

    return np.nan


if "R/W PUMP DUTY" in df.columns:
    df["R/W PUMP COUNT"] = df["R/W PUMP DUTY"].apply(pump_duty_to_count)
    print("已生成 R/W PUMP COUNT。")

if "T/W PUMP DUTY" in df.columns:
    df["T/W PUMP COUNT"] = df["T/W PUMP DUTY"].apply(pump_duty_to_count)
    print("已生成 T/W PUMP COUNT。")


non_numeric_cols = {
    "DATE", "Date", "date",
    "TIME", "Time", "time",
    "DATETIME", "OP_DATE",
    "REMARKS",
    "R/W PUMP DUTY",
    "T/W PUMP DUTY",
}

for col in df.columns:
    if col not in non_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")


if "F/RIDE" in df.columns:
    before = int(df["F/RIDE"].isna().sum())
    df["F/RIDE"] = df["F/RIDE"].fillna(0)
    after = int(df["F/RIDE"].isna().sum())
    print(f"F/RIDE 缺失填 0：before={before}, after={after}")

if CLIP_NTU:
    before_above = int((df[TARGET_COL] > NTU_CLIP_UPPER).sum())
    df[TARGET_COL] = df[TARGET_COL].clip(upper=NTU_CLIP_UPPER)
    after_above = int((df[TARGET_COL] > NTU_CLIP_UPPER).sum())
    print(f"NTU clipping: before_above={before_above}, after_above={after_above}")
else:
    print("未启用 NTU clipping。")

print("NTU 描述统计：")
display(df[TARGET_COL].describe())


## 6. 构造物理代理特征

第三题要求结合质量守恒 / 水力停留时间思想。  
由于附件没有完整池体体积和水力模型参数，本 baseline 构造三个代理特征：

```text
HRT_PROXY = C/W WELL LEVEL / T/W FLOW
FLOW_RATIO = R/W FLOW / T/W FLOW
WELL_LEVEL_CHANGE = C/W WELL LEVEL_t - C/W WELL LEVEL_{t-1}
```

含义：

```text
HRT_PROXY：清水池水力停留时间代理；
FLOW_RATIO：进出水流量平衡；
WELL_LEVEL_CHANGE：清水池水量变化趋势。
```


In [ ]:
# HRT proxy
if "C/W WELL LEVEL" in df.columns and "T/W FLOW" in df.columns:
    df["HRT_PROXY"] = df["C/W WELL LEVEL"] / (df["T/W FLOW"].replace(0, np.nan) + 1e-6)
else:
    df["HRT_PROXY"] = np.nan

# Flow ratio
if "R/W FLOW" in df.columns and "T/W FLOW" in df.columns:
    df["FLOW_RATIO"] = df["R/W FLOW"] / (df["T/W FLOW"].replace(0, np.nan) + 1e-6)
else:
    df["FLOW_RATIO"] = np.nan

# Well level change
if "C/W WELL LEVEL" in df.columns:
    df["WELL_LEVEL_CHANGE"] = df["C/W WELL LEVEL"].diff()
else:
    df["WELL_LEVEL_CHANGE"] = np.nan

physics_cols = ["HRT_PROXY", "FLOW_RATIO", "WELL_LEVEL_CHANGE"]

display(df[["DATETIME", TARGET_COL] + physics_cols].head(15))


## 7. 选择第三题过程变量

候选过程变量包括：

```text
FILT. NTU
R/W NTU
R/W PH
R/W FLOW
T/W FLOW
C/W WELL LEVEL
ALUM
F/RIDE
CL2
CLR
PH
RIVER LEVEL
R/W PUMP COUNT
T/W PUMP COUNT
HRT_PROXY
FLOW_RATIO
WELL_LEVEL_CHANGE
```

不存在的列会自动跳过。


In [ ]:
candidate_process_vars = [
    "FILT. NTU",
    "R/W NTU",
    "R/W PH",
    "R/W CLR",
    "RIVER LEVEL",
    "ALUM",
    "F/RIDE",
    "R/W FLOW",
    "T/W FLOW",
    "C/W WELL LEVEL",
    "18ML LEVEL",
    "18ML FLOW",
    "PH",
    "CLR",
    "CL2",
    "R/W PUMP COUNT",
    "T/W PUMP COUNT",
    "HRT_PROXY",
    "FLOW_RATIO",
    "WELL_LEVEL_CHANGE",
]

process_vars = [col for col in candidate_process_vars if col in df.columns]

print("实际使用过程变量数：", len(process_vars))
print(process_vars)

audit_cols = ["DATETIME", "OP_DATE", TARGET_COL] + process_vars
audit_cols = list(dict.fromkeys([c for c in audit_cols if c in df.columns]))

audit_df = pd.DataFrame({
    "column": audit_cols,
    "dtype": [str(df[c].dtype) for c in audit_cols],
    "missing_count": [int(df[c].isna().sum()) for c in audit_cols],
    "missing_rate": [float(df[c].isna().mean()) for c in audit_cols],
    "unique_count": [int(df[c].nunique(dropna=True)) for c in audit_cols],
}).sort_values("missing_rate", ascending=False)

audit_path = OUTPUT_DIR / "problem3_baseline_data_audit.xlsx"
audit_df.to_excel(audit_path, index=False)

print("数据审计表已保存：", audit_path)
display(audit_df)


## 8. 构造历史 lag 特征

规则：

```text
NTU：只构造 lag1-lag12，避免 target leakage；
过程变量：构造 lag0-lag12；
```

这样在时间 t 预测未来 NTU 时，可以使用时间 t 当前可观测的水厂状态和过去历史。


In [ ]:
feature_df = df.copy().sort_values("DATETIME").reset_index(drop=True)

created_feature_records = []

# NTU historical lags
ntu_lag_features = []
for lag in NTU_LAGS:
    col_name = f"{TARGET_COL}_lag{lag}"
    feature_df[col_name] = feature_df[TARGET_COL].shift(lag)
    ntu_lag_features.append(col_name)

    created_feature_records.append({
        "feature": col_name,
        "source_variable": TARGET_COL,
        "feature_type": "target_history_lag",
        "lag": lag,
        "lag_hours": lag * RECORD_INTERVAL_HOURS,
    })

# Process lags
process_lag_features = []
for var in process_vars:
    for lag in PROCESS_LAGS:
        col_name = f"{var}_lag{lag}"
        feature_df[col_name] = feature_df[var].shift(lag)
        process_lag_features.append(col_name)

        created_feature_records.append({
            "feature": col_name,
            "source_variable": var,
            "feature_type": "process_lag",
            "lag": lag,
            "lag_hours": lag * RECORD_INTERVAL_HOURS,
        })

feature_cols = ntu_lag_features + process_lag_features
feature_cols = list(dict.fromkeys([c for c in feature_cols if c in feature_df.columns]))

feature_summary_df = pd.DataFrame(created_feature_records)
feature_summary_path = OUTPUT_DIR / "problem3_baseline_feature_summary.xlsx"
feature_summary_df.to_excel(feature_summary_path, index=False)

print("NTU lag features:", len(ntu_lag_features))
print("Process lag features:", len(process_lag_features))
print("Total feature count:", len(feature_cols))
print("特征说明表已保存：", feature_summary_path)
display(feature_summary_df.head(30))


## 9. 构造未来多步预测目标

构造：

```text
target_NTU_t_plus_2h
target_NTU_t_plus_4h
target_NTU_t_plus_6h
target_NTU_t_plus_8h
target_NTU_t_plus_10h
target_NTU_t_plus_12h
```

每一行表示：

```text
用当前时间 t 的状态预测未来 2–12 小时 NTU。
```


In [ ]:
target_cols = []

for step, hour in zip(FORECAST_STEPS, HORIZON_HOURS):
    col_name = f"target_NTU_t_plus_{hour}h"
    feature_df[col_name] = feature_df[TARGET_COL].shift(-step)
    target_cols.append(col_name)

model_data_cols = ["DATETIME", "OP_DATE", TARGET_COL] + feature_cols + target_cols
model_data = feature_df[model_data_cols].copy()

# 删除没有完整未来目标的行
model_data = model_data.dropna(subset=["DATETIME"] + target_cols).sort_values("DATETIME").reset_index(drop=True)

model_data_path = OUTPUT_DIR / "problem3_baseline_model_data.xlsx"
model_data.to_excel(model_data_path, index=False)

print("建模数据规模：", model_data.shape)
print("建模数据已保存：", model_data_path)
display(model_data[["DATETIME", "OP_DATE", TARGET_COL] + target_cols].head())
display(model_data[["DATETIME", "OP_DATE", TARGET_COL] + target_cols].tail())


## 10. 时间顺序划分训练集和测试集

为避免未来信息泄漏，采用时间顺序划分：

```text
前 70%：train
后 30%：test
```


In [ ]:
n = len(model_data)
split_idx = int(n * TRAIN_RATIO)

train_df = model_data.iloc[:split_idx].copy()
test_df = model_data.iloc[split_idx:].copy()

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

Y_train = train_df[target_cols].values
Y_test = test_df[target_cols].values

print("总样本数：", n)
print("训练集：", train_df.shape, train_df["DATETIME"].min(), "至", train_df["DATETIME"].max())
print("测试集：", test_df.shape, test_df["DATETIME"].min(), "至", test_df["DATETIME"].max())
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)


## 11. 定义多步预测模型和评价函数

这里使用 Direct Multi-output 方式：

```text
输入 X_t
输出 [NTU_t+2h, ..., NTU_t+12h]
```

模型：

```text
Ridge
Random Forest
XGBoost
```

其中 XGBoost 需要 `MultiOutputRegressor` 包装。


In [ ]:
def evaluate_horizon_metrics(y_true_2d, y_pred_2d, model_name):
    records = []

    for idx, hour in enumerate(HORIZON_HOURS):
        y_true = y_true_2d[:, idx]
        y_pred = y_pred_2d[:, idx]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        mask = np.abs(y_true) > 1e-6
        if mask.sum() > 0:
            mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        else:
            mape = np.nan

        records.append({
            "model": model_name,
            "horizon_step": idx + 1,
            "horizon_hours": hour,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_percent": mape,
        })

    return pd.DataFrame(records)


def evaluate_overall_metrics(y_true_2d, y_pred_2d):
    y_true_flat = y_true_2d.reshape(-1)
    y_pred_flat = y_pred_2d.reshape(-1)

    return {
        "MAE": mean_absolute_error(y_true_flat, y_pred_flat),
        "RMSE": np.sqrt(mean_squared_error(y_true_flat, y_pred_flat)),
        "R2": r2_score(y_true_flat, y_pred_flat),
    }


def build_models():
    models = {}

    models["Ridge"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ])

    models["Random Forest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
        )),
    ])

    if XGB_AVAILABLE:
        xgb_base = XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", MultiOutputRegressor(xgb_base)),
        ])

    return models


print("模型和评价函数定义完成。")


## 12. 训练 baseline 多步预测模型

输出：

```text
problem3_baseline_model_results.xlsx
problem3_baseline_horizon_metrics.xlsx
problem3_baseline_test_predictions.xlsx
models/*.joblib
```


In [ ]:
models = build_models()

overall_records = []
horizon_metric_frames = []
prediction_frames = []
trained_models = {}

for model_name, model in models.items():
    print("=" * 80)
    print("Training model:", model_name)
    print("=" * 80)

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    overall_metrics = evaluate_overall_metrics(Y_test, Y_pred)

    overall_record = {
        "model": model_name,
        "n_features": len(feature_cols),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        **overall_metrics,
    }

    overall_records.append(overall_record)

    horizon_df = evaluate_horizon_metrics(Y_test, Y_pred, model_name)
    horizon_metric_frames.append(horizon_df)

    pred_df = test_df[["DATETIME", "OP_DATE", TARGET_COL]].copy()
    pred_df["model"] = model_name

    for idx, hour in enumerate(HORIZON_HOURS):
        pred_df[f"actual_NTU_t_plus_{hour}h"] = Y_test[:, idx]
        pred_df[f"predicted_NTU_t_plus_{hour}h"] = Y_pred[:, idx]
        pred_df[f"residual_NTU_t_plus_{hour}h"] = Y_test[:, idx] - Y_pred[:, idx]

    prediction_frames.append(pred_df)

    model_path = MODEL_DIR / f"problem3_baseline_{model_name}.joblib".replace(" ", "_")
    joblib.dump({
        "model": model,
        "feature_cols": feature_cols,
        "target_cols": target_cols,
        "horizon_hours": HORIZON_HOURS,
        "overall_metrics": overall_metrics,
    }, model_path)

    trained_models[model_name] = {
        "model": model,
        "overall_metrics": overall_metrics,
        "Y_pred": Y_pred,
        "model_path": model_path,
    }

    print(f"MAE={overall_metrics['MAE']:.6f}, RMSE={overall_metrics['RMSE']:.6f}, R2={overall_metrics['R2']:.6f}")

results_df = pd.DataFrame(overall_records).sort_values(["RMSE", "MAE"], ascending=[True, True]).reset_index(drop=True)
horizon_metrics_df = pd.concat(horizon_metric_frames, ignore_index=True)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

results_path = OUTPUT_DIR / "problem3_baseline_model_results.xlsx"
horizon_metrics_path = OUTPUT_DIR / "problem3_baseline_horizon_metrics.xlsx"
predictions_path = OUTPUT_DIR / "problem3_baseline_test_predictions.xlsx"

results_df.to_excel(results_path, index=False)
horizon_metrics_df.to_excel(horizon_metrics_path, index=False)
predictions_df.to_excel(predictions_path, index=False)

print("模型总体结果已保存：", results_path)
print("分 horizon 结果已保存：", horizon_metrics_path)
print("测试集预测结果已保存：", predictions_path)

display(results_df)
display(horizon_metrics_df)


## 13. 选择最佳 baseline 模型

按整体 RMSE 最小选择最佳模型。  
后续目标日期预测和敏感性分析默认使用该模型。


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]["model"]

print("最佳 baseline 模型：", best_model_name)
display(results_df.iloc[[0]])


## 14. 输出目标日期 07:00–19:00 预测结果

题目要求给出：

```text
2026-02-01
2026-02-10
2026-02-20
```

三个日期 07:00–19:00 的 NTU 预测结果。

本 baseline 采用以下逻辑：

```text
对于目标时刻 T，优先使用 T-2h 的样本作为 anchor，
用 horizon=2h 的预测作为 T 的预测结果。
如果 T-2h 不存在，则尝试 T-4h、T-6h ... T-12h。
```

输出：

```text
problem3_baseline_target_7_19_predictions.xlsx
```


In [ ]:
# 为所有可用 anchor 生成最佳模型预测
all_anchor_df = model_data[["DATETIME", "OP_DATE", TARGET_COL] + feature_cols].copy()
X_all = all_anchor_df[feature_cols].copy()
Y_all_pred = best_model.predict(X_all)

anchor_prediction_records = []

for i, row in all_anchor_df.iterrows():
    for idx, hour in enumerate(HORIZON_HOURS):
        target_datetime = row["DATETIME"] + pd.Timedelta(hours=hour)

        anchor_prediction_records.append({
            "anchor_DATETIME": row["DATETIME"],
            "anchor_OP_DATE": row["OP_DATE"],
            "horizon_hours": hour,
            "target_DATETIME": target_datetime,
            "predicted_NTU": Y_all_pred[i, idx],
            "model": best_model_name,
        })

anchor_pred_df = pd.DataFrame(anchor_prediction_records)

# 目标时间列表
target_datetime_records = []

for date_str in TARGET_DATES:
    for hour in TARGET_HOURS:
        target_dt = pd.to_datetime(date_str) + pd.Timedelta(hours=hour)
        target_datetime_records.append({
            "target_date": date_str,
            "target_hour": hour,
            "target_DATETIME": target_dt,
        })

target_times_df = pd.DataFrame(target_datetime_records)

# 对每个目标时刻，优先选择最短 horizon
target_prediction_records = []

for _, target_row in target_times_df.iterrows():
    target_dt = target_row["target_DATETIME"]

    candidates = anchor_pred_df[
        anchor_pred_df["target_DATETIME"] == target_dt
    ].copy()

    if len(candidates) == 0:
        target_prediction_records.append({
            "target_date": target_row["target_date"],
            "target_hour": target_row["target_hour"],
            "target_DATETIME": target_dt,
            "anchor_DATETIME": pd.NaT,
            "horizon_hours": np.nan,
            "predicted_NTU": np.nan,
            "model": best_model_name,
            "status": "NO_AVAILABLE_ANCHOR",
        })
        continue

    candidates = candidates.sort_values("horizon_hours", ascending=True)
    chosen = candidates.iloc[0]

    target_prediction_records.append({
        "target_date": target_row["target_date"],
        "target_hour": target_row["target_hour"],
        "target_DATETIME": target_dt,
        "anchor_DATETIME": chosen["anchor_DATETIME"],
        "horizon_hours": chosen["horizon_hours"],
        "predicted_NTU": chosen["predicted_NTU"],
        "model": best_model_name,
        "status": "OK",
    })

target_predictions_df = pd.DataFrame(target_prediction_records)

target_predictions_path = OUTPUT_DIR / "problem3_baseline_target_7_19_predictions.xlsx"
target_predictions_df.to_excel(target_predictions_path, index=False)

print("目标日期 07:00–19:00 预测结果已保存：", target_predictions_path)
display(target_predictions_df)


## 15. 敏感性分析

题目要求分析：

```text
原水水质突变
矾投加调整
```

本 baseline 使用目标日期预测时用到的 anchor 样本进行情景模拟：

```text
Scenario A：R/W NTU 相关特征增加 20%
Scenario B：ALUM 相关特征增加 10%
```

输出：

```text
problem3_baseline_sensitivity_analysis.xlsx
```


In [ ]:
def apply_feature_multiplier(X, feature_names, source_variable, multiplier):
    X_new = X.copy()

    # 匹配类似 "R/W NTU_lag0", "R/W NTU_lag1" 的特征
    prefix = f"{source_variable}_lag"

    matched_cols = [c for c in feature_names if c.startswith(prefix)]

    for c in matched_cols:
        if c in X_new.columns:
            X_new[c] = X_new[c] * multiplier

    return X_new, matched_cols


sensitivity_records = []

# 只对 status=OK 的目标时刻做敏感性
ok_targets = target_predictions_df[target_predictions_df["status"] == "OK"].copy()

for _, row in ok_targets.iterrows():
    anchor_dt = row["anchor_DATETIME"]
    horizon = int(row["horizon_hours"])
    horizon_idx = HORIZON_HOURS.index(horizon)

    anchor_feature_row = model_data[model_data["DATETIME"] == anchor_dt]

    if len(anchor_feature_row) == 0:
        continue

    X_base = anchor_feature_row[feature_cols].copy()

    base_pred_all = best_model.predict(X_base)
    base_pred = float(base_pred_all[0, horizon_idx])

    # Scenario A: R/W NTU +20%
    if "R/W NTU" in process_vars:
        X_raw_shock, raw_matched_cols = apply_feature_multiplier(
            X_base,
            feature_cols,
            "R/W NTU",
            1 + RAW_NTU_SHOCK_RATE,
        )
        raw_pred_all = best_model.predict(X_raw_shock)
        raw_pred = float(raw_pred_all[0, horizon_idx])
    else:
        raw_matched_cols = []
        raw_pred = np.nan

    # Scenario B: ALUM +10%
    if "ALUM" in process_vars:
        X_alum_adjust, alum_matched_cols = apply_feature_multiplier(
            X_base,
            feature_cols,
            "ALUM",
            1 + ALUM_ADJUST_RATE,
        )
        alum_pred_all = best_model.predict(X_alum_adjust)
        alum_pred = float(alum_pred_all[0, horizon_idx])
    else:
        alum_matched_cols = []
        alum_pred = np.nan

    sensitivity_records.append({
        "target_date": row["target_date"],
        "target_hour": row["target_hour"],
        "target_DATETIME": row["target_DATETIME"],
        "anchor_DATETIME": anchor_dt,
        "horizon_hours": horizon,
        "model": best_model_name,
        "baseline_predicted_NTU": base_pred,
        "raw_NTU_plus20_predicted_NTU": raw_pred,
        "raw_NTU_plus20_delta": raw_pred - base_pred if pd.notna(raw_pred) else np.nan,
        "alum_plus10_predicted_NTU": alum_pred,
        "alum_plus10_delta": alum_pred - base_pred if pd.notna(alum_pred) else np.nan,
        "raw_NTU_affected_feature_count": len(raw_matched_cols),
        "alum_affected_feature_count": len(alum_matched_cols),
    })

sensitivity_df = pd.DataFrame(sensitivity_records)

sensitivity_path = OUTPUT_DIR / "problem3_baseline_sensitivity_analysis.xlsx"
sensitivity_df.to_excel(sensitivity_path, index=False)

print("敏感性分析结果已保存：", sensitivity_path)
display(sensitivity_df)


## 16. 可视化：模型整体性能对比

输出：

```text
figures/problem3_baseline_model_rmse_comparison.png
figures/problem3_baseline_model_r2_comparison.png
```


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["model"], results_df["RMSE"])
plt.ylabel("Overall RMSE")
plt.title("Problem 3 Baseline Overall RMSE")
plt.tight_layout()

rmse_fig_path = FIG_DIR / "problem3_baseline_model_rmse_comparison.png"
plt.savefig(rmse_fig_path, dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results_df["model"], results_df["R2"])
plt.ylabel("Overall R²")
plt.title("Problem 3 Baseline Overall R²")
plt.tight_layout()

r2_fig_path = FIG_DIR / "problem3_baseline_model_r2_comparison.png"
plt.savefig(r2_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("RMSE 图已保存：", rmse_fig_path)
print("R² 图已保存：", r2_fig_path)


## 17. 可视化：不同 horizon 的误差

输出：

```text
figures/problem3_baseline_horizon_rmse.png
figures/problem3_baseline_horizon_r2.png
```


In [ ]:
plt.figure(figsize=(10, 5))

for model_name in horizon_metrics_df["model"].unique():
    temp = horizon_metrics_df[horizon_metrics_df["model"] == model_name]
    plt.plot(
        temp["horizon_hours"],
        temp["RMSE"],
        marker="o",
        label=model_name,
    )

plt.xlabel("Forecast horizon (hours)")
plt.ylabel("RMSE")
plt.title("Problem 3 Baseline RMSE by Horizon")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

horizon_rmse_fig_path = FIG_DIR / "problem3_baseline_horizon_rmse.png"
plt.savefig(horizon_rmse_fig_path, dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))

for model_name in horizon_metrics_df["model"].unique():
    temp = horizon_metrics_df[horizon_metrics_df["model"] == model_name]
    plt.plot(
        temp["horizon_hours"],
        temp["R2"],
        marker="o",
        label=model_name,
    )

plt.xlabel("Forecast horizon (hours)")
plt.ylabel("R²")
plt.title("Problem 3 Baseline R² by Horizon")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

horizon_r2_fig_path = FIG_DIR / "problem3_baseline_horizon_r2.png"
plt.savefig(horizon_r2_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Horizon RMSE 图已保存：", horizon_rmse_fig_path)
print("Horizon R² 图已保存：", horizon_r2_fig_path)


## 18. 可视化：最佳模型测试集预测曲线

默认绘制最佳模型在 2h horizon 上的预测曲线。  
如果需要，也可以改成 4h、6h 等 horizon。


In [ ]:
best_pred_df = predictions_df[predictions_df["model"] == best_model_name].copy()

plot_hour = 2
actual_col = f"actual_NTU_t_plus_{plot_hour}h"
pred_col = f"predicted_NTU_t_plus_{plot_hour}h"
resid_col = f"residual_NTU_t_plus_{plot_hour}h"

plt.figure(figsize=(14, 5))
plt.plot(
    best_pred_df["DATETIME"],
    best_pred_df[actual_col],
    label=f"Actual NTU t+{plot_hour}h",
    linewidth=1.5,
)
plt.plot(
    best_pred_df["DATETIME"],
    best_pred_df[pred_col],
    label=f"Predicted NTU t+{plot_hour}h",
    linewidth=1.5,
)

plt.title(f"Problem 3 Best Baseline Model Test Prediction: {best_model_name}, Horizon {plot_hour}h")
plt.xlabel("Anchor datetime")
plt.ylabel("NTU")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

best_ts_fig_path = FIG_DIR / "problem3_baseline_best_model_2h_timeseries.png"
plt.savefig(best_ts_fig_path, dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(
    best_pred_df["DATETIME"],
    best_pred_df[resid_col],
    linewidth=1.2,
)
plt.axhline(0, linestyle="--")
plt.title(f"Residuals: {best_model_name}, Horizon {plot_hour}h")
plt.xlabel("Anchor datetime")
plt.ylabel("Residual")
plt.grid(alpha=0.3)
plt.tight_layout()

residual_fig_path = FIG_DIR / "problem3_baseline_best_model_2h_residuals.png"
plt.savefig(residual_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("最佳模型 2h 预测曲线已保存：", best_ts_fig_path)
print("最佳模型 2h 残差图已保存：", residual_fig_path)


## 19. 输出汇总工作簿

汇总工作簿包含：

```text
model_results
horizon_metrics
target_predictions
sensitivity_analysis
feature_summary
data_audit
```

输出：

```text
problem3_baseline_summary.xlsx
```


In [ ]:
summary_workbook_path = OUTPUT_DIR / "problem3_baseline_summary.xlsx"

with pd.ExcelWriter(summary_workbook_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="model_results", index=False)
    horizon_metrics_df.to_excel(writer, sheet_name="horizon_metrics", index=False)
    target_predictions_df.to_excel(writer, sheet_name="target_predictions", index=False)
    sensitivity_df.to_excel(writer, sheet_name="sensitivity_analysis", index=False)
    feature_summary_df.to_excel(writer, sheet_name="feature_summary", index=False)
    audit_df.to_excel(writer, sheet_name="data_audit", index=False)

print("汇总工作簿已保存：", summary_workbook_path)


## 20. 最终摘要

运行到这里，第三题 baseline 已经完成。

论文中可以写：

```text
本文首先构建基于水力停留时间代理变量和历史序列特征的直接多步预测 baseline。模型以过去 NTU、滤后水浊度、进出水流量、清水池水位、投药量以及 HRT_PROXY、FLOW_RATIO、WELL_LEVEL_CHANGE 等物理代理特征为输入，直接输出未来 2–12 小时出厂水浊度。该 baseline 作为后续 GRU 序列模型的对照。
```


In [ ]:
final_summary = pd.DataFrame([
    {
        "item": "model_family",
        "value": "Direct Multi-step Hybrid Baseline",
    },
    {
        "item": "target",
        "value": "NTU",
    },
    {
        "item": "forecast_horizons",
        "value": ", ".join([f"{h}h" for h in HORIZON_HOURS]),
    },
    {
        "item": "physical_proxy_features",
        "value": "HRT_PROXY, FLOW_RATIO, WELL_LEVEL_CHANGE",
    },
    {
        "item": "best_model",
        "value": best_model_name,
    },
    {
        "item": "best_MAE",
        "value": results_df.iloc[0]["MAE"],
    },
    {
        "item": "best_RMSE",
        "value": results_df.iloc[0]["RMSE"],
    },
    {
        "item": "best_R2",
        "value": results_df.iloc[0]["R2"],
    },
    {
        "item": "target_prediction_file",
        "value": str(target_predictions_path),
    },
    {
        "item": "sensitivity_file",
        "value": str(sensitivity_path),
    },
])

final_summary_path = OUTPUT_DIR / "problem3_baseline_final_summary.xlsx"
final_summary.to_excel(final_summary_path, index=False)

print("=" * 80)
print("问题三 baseline 多步预测模型已完成。")
print("=" * 80)

print("\n核心输出文件：")
print("1. 数据审计：", audit_path)
print("2. 特征说明：", feature_summary_path)
print("3. 建模数据：", model_data_path)
print("4. 模型结果：", results_path)
print("5. 分 horizon 结果：", horizon_metrics_path)
print("6. 测试集预测：", predictions_path)
print("7. 目标日期预测：", target_predictions_path)
print("8. 敏感性分析：", sensitivity_path)
print("9. 汇总工作簿：", summary_workbook_path)
print("10. 最终摘要：", final_summary_path)
print("11. 图片目录：", FIG_DIR)
print("12. 模型目录：", MODEL_DIR)

print("\n模型结果：")
display(results_df)

print("\n目标日期预测：")
display(target_predictions_df)

print("\n敏感性分析：")
display(sensitivity_df)

print("\n最终摘要：")
display(final_summary)
